## 1) Pré-processamento: Limpeza dos dados

Primeiramente, temos que obter o dataset e verificar sua integridade.

In [7]:
import pandas as pd


dados = pd.read_csv('_ASSOC_VoleiStars.csv', encoding='latin1')
dados.info()

FileNotFoundError: [Errno 2] No such file or directory: '_ASSOC_VoleiStars.csv'

De acordo com o pdf, 
> O Conjunto é composto pelo número da partida, o nome do(a)s jogadore(a)spresentes, e o resultado da partida.

Contudo, percebe-se que existem no dataframe duas colunas sem nome e sem registro. Seria interessante remove-las para que não poluam o dataset.

In [ ]:
dados = dados.drop(columns=['Unnamed: 3', 'Unnamed: 4'])
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Partida         150 non-null    int64
 1   Jogadore(a)s    150 non-null    str  
 2   Resultado       150 non-null    str  
 3   Jogadore(a)s.1  8 non-null      str  
dtypes: int64(1), str(3)
memory usage: 4.8 KB


Existe também uma coluna que contém os nomes dos oito jogadores. Para fins de análise, removeremos essa coluna do dataframe, mas vamos guardar seus valores para limpeza e posterior transformações.

In [ ]:
jogadores = dados['Jogadore(a)s.1'].dropna()
dados = dados.drop(columns=['Jogadore(a)s.1'])
jogadores

0    Ricardo
1      Fábio
2      Ágata
3        Ana
4    Bárbara
5     Shelda
6    Sheldom
7    Emanuel
Name: Jogadore(a)s.1, dtype: str

Agora que a tabela contém apenas os dados relevantes, podemos analisar a integridade deles.

In [ ]:
dados.head()

,Partida,Jogadore(a)s,Resultado
0,1,"ricardo, fabio, emanuel",Perdeu
1,2,"Emanuel, fabio, ?gata",Perdeu
2,3,"?gata, Sheldom, Ana",GANHOU
3,4,"?gata, Ana, B rbara",GANHOU
4,5,"B rbara, F bio, Ricardo",Perdeu


Percebe-se que claramente o nome dos jogadores em algumas das fileiras está errado. Vamos ver quantas escritas erradas diferentes existem...

In [ ]:
nomes_errados = []

for time in dados['Jogadore(a)s']:
    for nome in time.split(','):
        nome = nome.strip()
        if nome not in nomes_errados:
            nomes_errados.append(nome)
            
nomes_errados.sort()
print(f'Número de nomes diferentes encontrados: {len(nomes_errados)}')
nomes_errados

Número de nomes diferentes encontrados: 17


['?gata',
 'Agata',
 'Ana',
 'Barbara',
 'B\xa0rbara',
 'Emanuel',
 'Fabio',
 'F\xa0bio',
 'Ricardo',
 'Shelda',
 'Sheldom',
 'ana',
 'emanuel',
 'fabio',
 'ricardo',
 'shelda',
 'sheldom']

Existem 17 nomes no total, 9 mais do que o esperado. Analisando esses nomes, três erros ficam aparentes:

1. Certos nomes são registrados duas vezes: uma vez em caixa alta, outra completamente minúsculo
2. A letra "Á" maiúscula é substituída por "?"
3. A letra "á" minúscula é substituída por "\xa0"

Dessa forma, a limpeza dos dados errôneos fica fácil. Vamos criar um dicionário que mapeia {nome_errado: nome_correto} para que seja possível transformar os dados posteriormente

In [ ]:
map_nome_errado_correto = {}

for nome_errado in nomes_errados:
    # obs: não substituimos "?" por "Á" nem "\xa0" por "á". Isso porque teriamos que tratar strings como "Ágata" e "Agata" como iguais, 
    # o que complicaria o código e não é relevante para fins de extração de padrões.
    nome_correto = nome_errado.capitalize().replace('?', 'A').replace('\xa0', 'a')
    map_nome_errado_correto[nome_errado] = nome_correto

# Pelo motivo mencionado acima, também vamos atualizar a lista dos nomes dos jogadores com os nomes sem acento
jogadores = sorted(list(set(map_nome_errado_correto.values())))
print(f'Nomes corretos: {jogadores}')
print(f'Número de nomes corretos: {len(jogadores)}')

map_nome_errado_correto

Nomes corretos: ['Agata', 'Ana', 'Barbara', 'Emanuel', 'Fabio', 'Ricardo', 'Shelda', 'Sheldom']
Número de nomes corretos: 8


{'?gata': 'Agata',
 'Agata': 'Agata',
 'Ana': 'Ana',
 'Barbara': 'Barbara',
 'B\xa0rbara': 'Barbara',
 'Emanuel': 'Emanuel',
 'Fabio': 'Fabio',
 'F\xa0bio': 'Fabio',
 'Ricardo': 'Ricardo',
 'Shelda': 'Shelda',
 'Sheldom': 'Sheldom',
 'ana': 'Ana',
 'emanuel': 'Emanuel',
 'fabio': 'Fabio',
 'ricardo': 'Ricardo',
 'shelda': 'Shelda',
 'sheldom': 'Sheldom'}

Agora que temos uma estrutura que mapea os nomes errados em nomes corretos, podemos fazer a transformação do dataframe original para que as regras de associação possam ser extraídas.

## 2) Transformação

Agora que temos uma forma de limpar os dados, podemos criar um dataframe com dados limpos e transformados para representação de one-hot encoding.

In [ ]:
linhas = []

for indice, linha in dados.iterrows():
    nova_linha = {}
    
    membros_da_equipe = linha['Jogadore(a)s'].split(',')
    for i in range(len(membros_da_equipe)):
        membros_da_equipe[i] = membros_da_equipe[i].strip()
        membros_da_equipe[i] = map_nome_errado_correto[membros_da_equipe[i]]
    
    for jogador in jogadores:
        nova_linha[jogador] = jogador in membros_da_equipe
    
    if linha['Resultado'].lower() == 'ganhou':
        nova_linha['Ganhou'] = True
        nova_linha['Perdeu'] = False
    else:
        nova_linha['Ganhou'] = False
        nova_linha['Perdeu'] = True
        
    linhas.append(nova_linha)
        
volei_stars = pd.DataFrame(linhas)
volei_stars.head(len(volei_stars))


NameError: name 'dados' is not defined

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Pega as transações mais frequentes, mas com suporte < 10% pra poder encontrar as combinações perdedoras também
transacoes_frequentes = apriori(volei_stars, min_support=0.05, use_colnames=True)

regras_de_associacao = association_rules(transacoes_frequentes, metric="confidence", min_threshold=0.5)

# Combinação vitoriosa: consequênte é "Ganhou", com maior confiança e supporte >= 10%
regras_de_vitoria = regras_de_associacao[
    (regras_de_associacao['consequents'] == frozenset({'Ganhou'})) & 
    (regras_de_associacao['support'] >= 0.10)
].sort_values(by=['confidence', 'support'], ascending=[False, False])

print("-=-=-=-=-=-=-=- Combinação Vitoriosa -=-=-=-=-=-=-=-")
print(regras_de_vitoria[['antecedents', 'support', 'confidence']].head(1))

# Combinação perdedora: consequente é "Peredu" com maior confiança
regras_de_perda = regras_de_associacao[
    (regras_de_associacao['consequents'] == frozenset({'Perdeu'}))
].sort_values(by=['confidence', 'support'], ascending=[False, False])

print("\n-=-=-=-=-=-=-=- Combinação Perdedora -=-=-=-=-=-=-=-")
print(regras_de_perda[['antecedents', 'support', 'confidence']].head(1))

# Estrela dos oito amigos
estrela = regras_de_vitoria[regras_de_vitoria['antecedents'].apply(lambda x: len(x) == 1)]

print("\n-=-=-=-=-=-=-=-  Estrela Dos Amigos  -=-=-=-=-=-=-=-")
print(estrela[['antecedents', 'support', 'confidence']].head(1))

NameError: name 'volei_stars' is not defined